In [1]:
from communication import CollDevice, Communicator, CollAlgo
from functools import reduce
import numpy as np
from itertools import product
from goal import GoalParallel, GoalSequential

DP_SIZE = 4
PP_SIZE = 3
TP_SIZE = 4
EP_SIZE = 4

GPU_GRID = (PP_SIZE, DP_SIZE, EP_SIZE, TP_SIZE)

In [2]:
devices = [CollDevice(i) for i in range(reduce(lambda x, y: x * y, GPU_GRID))]

In [3]:
devices_arr = np.array(devices).reshape(GPU_GRID)

In [4]:
comms = []
for i in range(len(GPU_GRID)):
    comms.append(
        np.broadcast_to(
            np.expand_dims(
                np.apply_along_axis(
                    lambda x: Communicator(x),
                    axis=i, arr=devices_arr
                    ),
                axis=i
            ),
            GPU_GRID
        )
    )

In [5]:
batch_seq_size = 4096
token_size = 2048
hidden_size = 4096
tp_compute_time = 0

In [6]:
goals = {}
context_dict = {
    "pp": 4,
    "dp": 3,
    "ep": 2,
    "tp": 1,
}
device2goal_rank = {ele: ele.device_id for ele in devices}
for pp, dp, ep, tp in product(range(PP_SIZE), range(DP_SIZE), range(EP_SIZE), range(TP_SIZE)):
    comm = [comms[i][pp, dp, ep, tp] for i in range(len(GPU_GRID))]
    pp_activation_size = batch_seq_size * token_size * 2 # assume 2 bytes per element
    with devices_arr[pp, dp, ep, tp] as device:
        if pp == 0:
            # a simple send to the next pp rank
            goals[device] = comm[0].send(pp_activation_size, context_dict["pp"], pp + 1).to_goal(device2goal_rank, 0, 0)
        elif pp == PP_SIZE - 1:
            goals[device] = comm[0].recv(pp_activation_size, context_dict["pp"], pp - 1).to_goal(device2goal_rank, 0, 0)
        else:
            weight_size = token_size * hidden_size * 2 # assume 2 bytes per element
            goal_pp = GoalParallel(
                [comm[0].send(pp_activation_size, context_dict["pp"], pp + 1).to_goal(device2goal_rank, 0, 0),
                 comm[0].recv(pp_activation_size, context_dict["pp"], pp - 1).to_goal(device2goal_rank, 0, 0)],
                cpu = 0
            )
            goal_dp = GoalSequential(
                [comm[1].allgather(weight_size, context_dict["dp"], CollAlgo.RECURSIVE_DOUBLING).to_goal(device2goal_rank, 1, 0),
                 comm[1].allgather(weight_size, context_dict["dp"], CollAlgo.RECURSIVE_DOUBLING).to_goal(device2goal_rank, 1, 0)],
                 cpu = 1
            ) # for the weights in the ffn
            goal_tp_ep = GoalSequential(
                [comm[2].alltoall(pp_activation_size, context_dict["ep"]).to_goal(device2goal_rank, 2, 0),
                 comm[3].allreduce(pp_activation_size, context_dict["tp"], CollAlgo.RECURSIVE_DOUBLING).to_goal(device2goal_rank, 2, 0),
                 comm[2].alltoall(pp_activation_size, context_dict["ep"]).to_goal(device2goal_rank, 2, 0)],
                 cpu = 2
            )
            goal_overall = GoalParallel([goal_pp, goal_dp, goal_tp_ep], cpu = 3)
            goals[device] = goal_overall

In [7]:
with open("minimal_trace.goal", "w") as f:
    f.write(f"num_ranks {len(devices)}\n")
    for device, goal in goals.items():
        with device:
            f.write(f"rank {device.device_id} {{\n")
            for line in goal.generate_lines():
                f.write(f"{line}\n")
            f.write("}\n")